In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image
img = cv2.imread("/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/odm_orthophoto.tif")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Smooth to get density-like variation
blur = cv2.GaussianBlur(gray, (51, 51), 0)

# Normalize
norm = cv2.normalize(blur, None, 0, 255, cv2.NORM_MINMAX)

# Apply heatmap
heatmap = cv2.applyColorMap(norm, cv2.COLORMAP_JET)

# Overlay
overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis("off")


In [ ]:
overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)


In [ ]:
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# =========================
# INPUT FILE PATHS
# =========================
JAN_TIF = "/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_21_6.tif"
FEB_TIF = "/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_8_7.tif"
MAR_TIF = "/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_10_8.tif"


GAUSSIAN_KERNEL = 51
CHANGE_THRESHOLD = 0.5

# =========================
# FUNCTIONS
# =========================

def load_rgb(tif_path):
    with rasterio.open(tif_path) as src:
        img = src.read([1, 2, 3])
        img = np.transpose(img, (1, 2, 0))
    return img.astype(np.float32)

def canopy_variation(rgb):
    gray = cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    smooth = gaussian_filter(gray, sigma=GAUSSIAN_KERNEL / 6)
    z = (smooth - np.mean(smooth)) / np.std(smooth)
    return z

def crop_to_common_extent(a, b, c):
    min_rows = min(a.shape[0], b.shape[0], c.shape[0])
    min_cols = min(a.shape[1], b.shape[1], c.shape[1])
    return (
        a[:min_rows, :min_cols],
        b[:min_rows, :min_cols],
        c[:min_rows, :min_cols]
    )

def classify_change(diff, th):
    stable = np.abs(diff) < th
    increase = diff >= th
    decrease = diff <= -th
    return stable, increase, decrease

def percent(mask):
    return 100 * np.sum(mask) / mask.size

# =========================
# LOAD ORTHOMOSAICS
# =========================
print("Loading orthomosaics...")
jan_rgb = load_rgb(JAN_TIF)
feb_rgb = load_rgb(FEB_TIF)
mar_rgb = load_rgb(MAR_TIF)

# =========================
# CANOPY VARIATION MAPS
# =========================
print("Computing canopy variation maps...")
jan_can = canopy_variation(jan_rgb)
feb_can = canopy_variation(feb_rgb)
mar_can = canopy_variation(mar_rgb)

# =========================
# FIX SHAPE MISMATCH
# =========================
jan_can, feb_can, mar_can = crop_to_common_extent(
    jan_can, feb_can, mar_can
)

print("Common shape:", jan_can.shape)

# =========================
# TEMPORAL DIFFERENCES
# =========================
diff_feb_jan = feb_can - jan_can
diff_mar_feb = mar_can - feb_can

# =========================
# CHANGE CLASSIFICATION
# =========================
stable_fj, inc_fj, dec_fj = classify_change(diff_feb_jan, CHANGE_THRESHOLD)
stable_mf, inc_mf, dec_mf = classify_change(diff_mar_feb, CHANGE_THRESHOLD)

# =========================
# TEMPORAL VARIANCE
# =========================
stack = np.stack([jan_can, feb_can, mar_can])
temporal_variance = np.var(stack, axis=0)

# =========================
# METRICS
# =========================
print("\n===== CANOPY CHANGE (%) =====")
print(f"Feb–Jan  Increase: {percent(inc_fj):.2f}")
print(f"Feb–Jan  Decrease: {percent(dec_fj):.2f}")
print(f"Feb–Jan  Stable  : {percent(stable_fj):.2f}")

print(f"\nMar–Feb  Increase: {percent(inc_mf):.2f}")
print(f"Mar–Feb  Decrease: {percent(dec_mf):.2f}")
print(f"Mar–Feb  Stable  : {percent(stable_mf):.2f}")

print("\n===== GLOBAL CANOPY METRICS =====")
print(f"Jan  mean/std: {np.mean(jan_can):.3f} / {np.std(jan_can):.3f}")
print(f"Feb  mean/std: {np.mean(feb_can):.3f} / {np.std(feb_can):.3f}")
print(f"Mar  mean/std: {np.mean(mar_can):.3f} / {np.std(mar_can):.3f}")
print(f"Temporal variance (mean): {np.mean(temporal_variance):.3f}")

# =========================
# VISUALIZATION
# =========================
fig, ax = plt.subplots(3, 3, figsize=(15, 15))

ax[0,0].imshow(jan_can, cmap="viridis")
ax[0,0].set_title("January canopy")

ax[0,1].imshow(feb_can, cmap="viridis")
ax[0,1].set_title("February canopy")

ax[0,2].imshow(mar_can, cmap="viridis")
ax[0,2].set_title("March canopy")

ax[1,0].imshow(diff_feb_jan, cmap="bwr", vmin=-2, vmax=2)
ax[1,0].set_title("Feb – Jan difference")

ax[1,1].imshow(diff_mar_feb, cmap="bwr", vmin=-2, vmax=2)
ax[1,1].set_title("Mar – Feb difference")

ax[1,2].imshow(temporal_variance, cmap="inferno")
ax[1,2].set_title("Temporal variance")

ax[2,0].imshow(inc_fj, cmap="Greens")
ax[2,0].set_title("Increase (Feb–Jan)")

ax[2,1].imshow(dec_fj, cmap="Reds")
ax[2,1].set_title("Decrease (Feb–Jan)")

ax[2,2].imshow(stable_fj, cmap="gray")
ax[2,2].set_title("Stable (Feb–Jan)")

for a in ax.flat:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# =========================
# INPUT
# =========================
ORTHO_TIF = "/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_8_7.tif"

SMOOTH_SCALE = 51   # increase for broader canopy patterns

# =========================
# LOAD IMAGE
# =========================
with rasterio.open(ORTHO_TIF) as src:
    img = src.read([1, 2, 3])
    img = np.transpose(img, (1, 2, 0))

img = img.astype(np.uint8)

# =========================
# BASIC TRANSFORMS
# =========================
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
smooth = gaussian_filter(gray, sigma=SMOOTH_SCALE / 6)

# Normalize
norm = (smooth - smooth.min()) / (smooth.max() - smooth.min())

# =========================
# 1️⃣ CANOPY INTENSITY MAP
# =========================
canopy_intensity = norm

# =========================
# 2️⃣ CANOPY DENSITY INDEX
# =========================
canopy_density = gaussian_filter(norm, sigma=SMOOTH_SCALE / 4)

# =========================
# 3️⃣ TEXTURE / HETEROGENEITY
# =========================
texture = cv2.Laplacian(gray, cv2.CV_32F)
texture = np.abs(texture)
texture = gaussian_filter(texture, sigma=SMOOTH_SCALE / 6)
texture = (texture - texture.min()) / (texture.max() - texture.min())

# =========================
# 4️⃣ GAP LIKELIHOOD
# =========================
gap_likelihood = 1 - canopy_density

# =========================
# 5️⃣ CROWN CLUSTERING
# =========================
crown_clusters = gaussian_filter(norm**2, sigma=SMOOTH_SCALE / 3)
crown_clusters = (crown_clusters - crown_clusters.min()) / (
    crown_clusters.max() - crown_clusters.min()
)

# =========================
# 6️⃣ EDGE / TRANSITION ZONES
# =========================
edges = cv2.Canny(gray, 50, 150)
edges = gaussian_filter(edges.astype(float), sigma=SMOOTH_SCALE / 6)

# =========================
# VISUALIZATION
# =========================
fig, ax = plt.subplots(2, 3, figsize=(18, 12))

ax[0,0].imshow(canopy_intensity, cmap="viridis")
ax[0,0].set_title("Canopy Intensity")

ax[0,1].imshow(canopy_density, cmap="YlGn")
ax[0,1].set_title("Canopy Density Index")

ax[0,2].imshow(texture, cmap="inferno")
ax[0,2].set_title("Structural Heterogeneity")

ax[1,0].imshow(gap_likelihood, cmap="Blues")
ax[1,0].set_title("Gap Likelihood")

ax[1,1].imshow(crown_clusters, cmap="hot")
ax[1,1].set_title("Crown Clustering")

ax[1,2].imshow(edges, cmap="gray")
ax[1,2].set_title("Canopy Transition Zones")

for a in ax.flat:
    a.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import cv2
import rasterio
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter, label

# =========================
# INPUT
# =========================
ORTHO_TIF = "/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_21_6.tif"

DENSE_THRESHOLD = 0.20     # dense canopy definition
SMOOTH_SCALE = 51          # spatial scale

# =========================
# LOAD ORTHOMOSAIC
# =========================
with rasterio.open(ORTHO_TIF) as src:
    img = src.read([1, 2, 3])
    img = np.transpose(img, (1, 2, 0)).astype(np.uint8)

# =========================
# CANOPY METRICS
# =========================
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

smooth = gaussian_filter(gray, sigma=SMOOTH_SCALE / 6)
canopy_density = (smooth - smooth.min()) / (smooth.max() - smooth.min())

texture = cv2.Laplacian(gray, cv2.CV_32F)
texture = np.abs(texture)
texture = gaussian_filter(texture, sigma=SMOOTH_SCALE / 6)
texture = (texture - texture.min()) / (texture.max() - texture.min())

# =========================
# DENSE CANOPY MASK
# =========================
dense_mask = canopy_density >= DENSE_THRESHOLD

# =========================
# LABEL DENSE PATCHES
# =========================
labels, num_patches = label(dense_mask)

# =========================
# EXTRACT PATCH METRICS
# =========================
patch_data = []

for patch_id in range(1, num_patches + 1):
    mask = labels == patch_id
    area = np.sum(mask)
    mean_density = np.mean(canopy_density[mask])
    mean_texture = np.mean(texture[mask])

    patch_data.append({
        "patch_id": patch_id,
        "area_pixels": area,
        "mean_density": mean_density,
        "mean_texture": mean_texture
    })

df = pd.DataFrame(patch_data)

# =========================
# EXPORT TO CSV
# =========================
csv_path = "dense_canopy_patches.csv"
df.to_csv(csv_path, index=False)

print(f"CSV exported: {csv_path}")
print(f"Number of dense canopy patches: {num_patches}")

# =========================
# GRAPH 1: PATCH SIZE DISTRIBUTION
# =========================
plt.figure(figsize=(6,4))
plt.hist(df["area_pixels"], bins=30, color="darkgreen", alpha=0.8)
plt.xlabel("Patch area (pixels)")
plt.ylabel("Number of patches")
plt.title("Dense canopy patch size distribution")
plt.show()

# =========================
# GRAPH 2: PATCH DENSITY vs TEXTURE
# =========================
plt.figure(figsize=(5,5))
plt.scatter(
    df["mean_density"],
    df["mean_texture"],
    s=20,
    alpha=0.7,
    color="forestgreen"
)
plt.xlabel("Mean canopy density")
plt.ylabel("Mean structural heterogeneity")
plt.title("Dense canopy patch characteristics")
plt.show()

# =========================
# GRAPH 3: TOTAL AREA CONTRIBUTION
# =========================
total_area = np.sum(canopy_density.size)
dense_area = np.sum(dense_mask)

plt.figure(figsize=(4,4))
plt.bar(
    ["Dense canopy", "Other"],
    [dense_area, total_area - dense_area],
    color=["darkgreen", "lightgray"]
)
plt.ylabel("Pixel count")
plt.title("Area contribution of dense canopy")
plt.show()

# =========================
# PRINT SUMMARY
# =========================
print("\n===== DENSE CANOPY SUMMARY =====")
print(f"Dense canopy area (%): {100 * dense_area / total_area:.2f}")
print(f"Mean dense patch size (pixels): {df['area_pixels'].mean():.1f}")
print(f"Mean dense patch density: {df['mean_density'].mean():.3f}")
print(f"Mean dense patch texture: {df['mean_texture'].mean():.3f}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =========================
# LOAD IMAGE
# =========================
img = cv2.imread("/Users/Shared/Files From d.localized/Guide_IITD/data_set_sanjayvan/Data/sanjay_van/Sanjay_Van_Drone/spot1_8_10/spot 2/s2_21_6.tif")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# =========================
# HEATMAP GENERATION (YOUR METHOD)
# =========================
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (51, 51), 0)
norm = cv2.normalize(blur, None, 0, 255, cv2.NORM_MINMAX)

heatmap = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

# =========================
# CANOPY CLASSIFICATION
# =========================
blue_mask   = norm <= 64
green_mask  = (norm > 64) & (norm <= 128)
yellow_mask = (norm > 128) & (norm <= 192)
red_mask    = norm > 192

areas = {
    "Open / gaps (Blue)": np.sum(blue_mask),
    "Moderate canopy (Green)": np.sum(green_mask),
    "Dense canopy (Yellow)": np.sum(yellow_mask),
    "Very dense canopy (Red)": np.sum(red_mask)
}

total_pixels = norm.size
area_percent = {k: (v / total_pixels) * 100 for k, v in areas.items()}

# =========================
# PRINT RESULTS
# =========================
print("\n===== CANOPY AREA SUMMARY =====")
for k in areas:
    print(f"{k}: {area_percent[k]:.2f} %")

# =========================
# PLOT BAR GRAPH
# =========================
plt.figure(figsize=(7,5))
plt.bar(
    areas.keys(),
    area_percent.values(),
    color=["royalblue", "green", "gold", "red"]
)
plt.ylabel("Area (%)")
plt.title("Canopy area distribution from heatmap")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# =========================
# SHOW HEATMAP OVERLAY
# =========================
plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Spot2(10/08/2025) Canopy heatmap overlay")
plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
from pathlib import Path
import re

# =========================
# INPUT ORTHOMOSAICS (AUTO-DISCOVER 7 OMs)
# =========================
om_dir = Path("../../input/input_om").resolve()
tif_paths = sorted(
    [p for p in om_dir.glob("*.tif") if ".aux." not in p.name],
    key=lambda p: int(re.search(r"(\d+)", p.stem).group(1)) if re.search(r"(\d+)", p.stem) else 999
 )

orthos = {f"OM{idx+1}": str(p) for idx, p in enumerate(tif_paths)}

print(f"Discovered {len(orthos)} orthomosaics from {om_dir}")
for k, v in orthos.items():
    print(f"  {k}: {Path(v).name}")

if len(orthos) != 7:
    print(f"⚠️ Expected 7 orthomosaics, found {len(orthos)}")

# =========================
# CANOPY BINS (JET)
# =========================
bins = {
    "Open / gaps (Blue)": (0, 64),
    "Moderate canopy (Green)": (65, 128),
    "Dense canopy (Yellow)": (129, 192),
    "Very dense canopy (Red)": (193, 255),
}

results = []

# =========================
# PROCESS EACH ORTHOMOSAIC
# =========================
for label, path in orthos.items():
    with rasterio.open(path) as src:
        img = src.read([1, 2, 3])
        img = np.transpose(img, (1, 2, 0))
        transform = src.transform

        # Pixel area in m²
        pixel_area_m2 = abs(transform.a * transform.e)

    img = img.astype(np.uint8)

    # Heatmap-style grayscale normalization
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (51, 51), 0)
    norm = cv2.normalize(blur, None, 0, 255, cv2.NORM_MINMAX)

    total_pixels = norm.size
    total_area_m2 = total_pixels * pixel_area_m2

    for cls, (low, high) in bins.items():
        mask = (norm >= low) & (norm <= high)
        pixel_count = int(np.sum(mask))

        area_m2 = pixel_count * pixel_area_m2
        area_ha = area_m2 / 10_000

        results.append({
            "date": label,
            "class": cls,
            "pixels": pixel_count,
            "area_m2": area_m2,
            "area_ha": area_ha,
            "area_percent": (area_m2 / total_area_m2) * 100,
        })

# =========================
# EXPORT TO CSV
# =========================
df = pd.DataFrame(results)
csv_path = Path("../../output/canopy_area_comparison_7om.csv").resolve()
df.to_csv(csv_path, index=False)

print(f"CSV exported: {csv_path}")

# =========================
# BAR PLOT COMPARISON
# =========================
pivot = df.pivot(index="class", columns="date", values="area_percent")

# Keep OM order OM1..OM7 when plotting
ordered_cols = sorted(pivot.columns, key=lambda x: int(x.replace("OM", "")))
pivot = pivot[ordered_cols]

pivot.plot(
    kind="bar",
    figsize=(13, 6),
    colormap="tab10",
    edgecolor="black",
    linewidth=0.3,
 )

plt.ylabel("Area (%)")
plt.title("Canopy class comparison across 7 orthomosaics")
plt.xticks(rotation=20)
plt.legend(title="Orthomosaic", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Pivot data for plotting
line_df = df.pivot(index="date", columns="class", values="area_percent")

# Plot
plt.figure(figsize=(9,5))

for cls in line_df.columns:
    plt.plot(
        line_df.index,
        line_df[cls],
        marker="o",
        linewidth=2,
        label=cls
    )

plt.xlabel("Date")
plt.ylabel("Area (%)")
plt.title("Temporal trend of canopy classes")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare stacked data
stack_df = df.pivot(index="date", columns="class", values="area_percent")

# Plot stacked bar chart
stack_df.plot(
    kind="bar",
    stacked=True,
    figsize=(9,5),
    color=["royalblue", "seagreen", "gold", "red"]
)

plt.ylabel("Area (%)")
plt.title("Canopy composition by date")
plt.legend(title="Canopy class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
#Patch size distribution
from scipy.ndimage import label

# use dense+moderate canopy as "canopy"
canopy_mask = norm > 128

labels, num = label(canopy_mask)

patch_sizes = [np.sum(labels == i) for i in range(1, num+1)]

plt.figure(figsize=(6,4))
plt.hist(patch_sizes, bins=30, log=True, color="teal")
plt.xlabel("Patch size (pixels)")
plt.ylabel("Frequency (log scale)")
plt.title("Canopy patch size distribution")
plt.show()


In [ ]:
#area vs gap intensity

gap_intensity = 1 - norm/255.0  # inverse of heatmap

# classify gap strength
low_gap = gap_intensity < 0.33
medium_gap = (gap_intensity >= 0.33) & (gap_intensity < 0.66)
high_gap = gap_intensity >= 0.66

areas = [
    np.sum(low_gap),
    np.sum(medium_gap),
    np.sum(high_gap)
]

plt.figure(figsize=(5,4))
plt.bar(
    ["Low gap", "Medium gap", "High gap"],
    areas,
    color=["lightgreen", "orange", "brown"]
)
plt.ylabel("Pixel count")
plt.title("Gap intensity distribution")
plt.show()
